In [ ]:
!nvidia-smi

Sun May 17 11:51:21 2026       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 457.49       Driver Version: 457.49       CUDA Version: 11.1     |
|-------------------------------+----------------------+----------------------+
| GPU  Name            TCC/WDDM | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  GeForce GTX 1050   WDDM  | 00000000:01:00.0  On |                  N/A |
| N/A   50C    P8    N/A /  N/A |    298MiB /  3072MiB |      5%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
                                                                               
+-------

In [ ]:
import torch
print(torch.cuda.is_available())

True


In [ ]:
from datasets import load_dataset

dataset = load_dataset("wandb/finqa-data-processed")

train_data = dataset["train"]
test_data = dataset["test"]

c:\Users\r2000\anaconda3\envs\fingpt\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Repo card metadata block was not found. Setting CardData to empty.


In [ ]:
train_data[0]

{'query': 'did abiomed outperform the nasdaq medical equipment index over the five year period?',
 'context': "Pre-text: ['performance graph the following graph compares the yearly change in the cumulative total stockholder return for our last five full fiscal years , based upon the market price of our common stock , with the cumulative total return on a nasdaq composite index ( u.s .', 'companies ) and a peer group , the nasdaq medical equipment-sic code 3840-3849 index , which is comprised of medical equipment companies , for that period .', 'the performance graph assumes the investment of $ 100 on march 31 , 2010 in our common stock , the nasdaq composite index ( u.s .', 'companies ) and the peer group index , and the reinvestment of any and all dividends. .']\n\n--\n\nTabular Data:\n----------------------------------------\nRow 1: , 3/31/2010, 3/31/2011, 3/31/2012, 3/31/2013, 3/31/2014, 3/31/2015\nRow 2: abiomed inc, 100, 140.79, 215.02, 180.91, 252.33, 693.60\nRow 3: nasdaq compos

In [ ]:
def format_example(example):
    prompt = f"""You are a financial reasoning assistant.

Context:
{example['context']}

Question:
{example['query']}

Answer:"""

    return {
        "text": prompt + " " + str(example["output"])
    }

train_data = train_data.map(format_example)
test_data = test_data.map(format_example)

In [ ]:
# from transformers import AutoModelForCausalLM, AutoTokenizer
# import torch

# model_name = "microsoft/phi-2"

# tokenizer = AutoTokenizer.from_pretrained(model_name)
# tokenizer.pad_token = tokenizer.eos_token

# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     load_in_4bit=True,   # QLoRA
#     device_map="auto",
#     torch_dtype=torch.float16
# )

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/tinyllama-bnb-4bit",
    max_seq_length = 512,        # VERY IMPORTANT (keep small)
    load_in_4bit = True,
    dtype = torch.float16,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W0514 23:03:54.614000 25544 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce GTX 1050. Num GPUs = 1. Max memory: 3.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.7.1+cu118. CUDA: 6.1. CUDA Toolkit: 11.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 201/201 [00:02<00:00, 94.43it/s] 
Unsloth: Will load unsloth/tinyllama-bnb-4bit as a legacy tokenizer.


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 8,                       # ↓ reduce from 16 → 8
    lora_alpha = 16,
    target_modules = ["q_proj", "v_proj"],
    lora_dropout = 0.05,
    use_gradient_checkpointing = True,
)

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.4.8 patched 22 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    per_device_train_batch_size = 1,   # MUST
    gradient_accumulation_steps = 8,   # simulate batch=8
    num_train_epochs = 2,
    learning_rate = 2e-4,
    fp16 = True,
    logging_steps = 20,
    save_steps = 200,
    output_dir = "finqa-small",
)

In [ ]:
train_data = train_data.select(range(2000))   # don't use full dataset initially

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model = model,
    train_dataset = train_data,
    tokenizer = tokenizer,
    dataset_text_field = "text",
    max_seq_length = 256,
    args = training_args,
)

trainer.train()

Unsloth: Tokenizing ["text"]: 100%|██████████| 2000/2000 [00:06<00:00, 299.64 examples/s]
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,000 | Num Epochs = 2 | Total steps = 500
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 1,126,400 of 1,101,174,784 (0.10% trained)


RuntimeError: Unsloth: No or negligible GPU memory available for fused cross entropy.

In [ ]:
import re

def normalize_answer(s):
    s = s.lower()
    s = re.sub(r"[^\w\s.%]", "", s)   # keep numbers, %, words
    return s.strip()

def exact_match(pred, truth):
    return normalize_answer(pred) == normalize_answer(truth)

def evaluate_model(model, dataset, tokenizer):
    correct = 0
    total = 0

    for sample in dataset.select(range(200)):   # small eval set
        prompt = f"""
Context:
{sample['context']}

Question:
{sample['query']}

Answer:
"""

        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

        outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            do_sample=False
        )

        pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
        pred = pred.split("Answer:")[-1].strip()

        gt = str(sample["output"]).strip()

        if exact_match(pred, gt):
            correct += 1

        total += 1

    accuracy = correct / total
    return accuracy

In [ ]:
acc = evaluate_model(model, test_data, tokenizer)
print(f"Exact Match Accuracy: {acc:.2f}")

Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Unsloth: Input IDs of shape torch.Size([1, 1132]) with length 1132 > the model's max sequence length of 512.
We shall truncate it ourselves. It's imperative if you correct this issue first.
c:\Users\r2000\anaconda3\envs\fingpt\Lib\site-packages\transformers\modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
c:\Users\r2000\anaconda3\envs\fingpt\Lib\site-packages\transformers\modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils`

RuntimeError: Triton Error [CUDA]: device kernel image is invalid